# Kidney dataset selection — audit (`_repaired`)

Ten notebook audytuje **już wyselekcjonowany i pobrany** korpus kidney (`data/kidney_workspace/configs/datasets/kidney/`, 30 datasetów, 30 ręcznych wykluczeń z poprzedniej rundy) tą samą, zwalidowaną na brain metodyką: obiektywna detekcja duplikatów technicznych, warianty kalibracyjne (`null_mz_shift`), fragmenty mikroanatomiczne, ekstremalnie małe akwizycje.

**Świadomie POMINIĘTE w tym notebooku (na wyraźną prośbę):**

- Analiza wspólnego zakresu m/z (Etap 1–3 z `brain_dataset.ipynb`, sekcja 5). Ujednolicenie osi m/z między narządami będzie rozwiązane osobno, później — tu zostawiam bez zmian dotychczasowy, już przyjęty zakres kidney (`mz_min=200, mz_max=900`).

**Utrzymane ograniczenia z `brain_dataset.ipynb`:**

- Wyłącznie `msi_dataset_manager.exploration.DatasetExplorer` — zero zmian w bibliotece.
- Zero nowych pobrań surowych danych (30 datasetów kidney jest już na dysku; ten notebook tylko odpytuje metadane METASPACE do audytu i eksportuje poprawioną konfigurację).

In [1]:
import os
from pathlib import Path

current_path = Path.cwd().resolve()
repository_root = next(
    path
    for path in (current_path, *current_path.parents)
    if (path / "pyproject.toml").is_file()
)
os.chdir(repository_root)

repository_root

PosixPath('/home/max/repositories/MSIAutoEncoderWrapper')

In [2]:
import json
import re

import pandas as pd
from IPython.display import display

from msi_dataset_manager.exploration import DatasetExplorer

# REMARK: date i download DB is 12.08.2026 (DD, MM, YYYY) -- same cache as brain_dataset.ipynb.
explorer = DatasetExplorer(
    source="metaspace",
    cache_dir="assets/local/datasets/metaspace",
    refresh_cache=False,
)

## 1. Szeroka pula kandydatów (ten sam filtr biologiczny co dotychczasowy `kidney_dataset.ipynb`)

W przeciwieństwie do brain (gdzie użyłem `condition=["Wildtype", "Wtype", "N/A"]`), tutaj zachowuję dokładnie filtr, który wygenerował obecny korpus kidney: `condition="Wildtype"` (bez wariantów pisowni i bez `N/A`) — nie zmieniam cicho zakresu biologicznego już przyjętego dla tego narządu. Zapytanie jest celowo **bez** `mz_min`/`mz_max`, żeby audyt duplikatów objął też rekordy, które istniejący filtr `200–900` odrzuca (przydatne np. gdybyś kiedyś chciał poszerzyć zakres).

In [3]:
broad_filters = {
    "organism": "Mouse",
    "organism_part": "Kidney",
    "condition": "Wildtype",
    "polarity": "Negative",
    "annotation_fdr": 0.1,
    "min_annotation_count": 1,
}
results = explorer.filter(broad_filters)
print(f"Found {len(results)} datasets")
display(results[["dataset_id", "name", "analyzer_type", "ionisation_source", "mz_min", "mz_max", "pixel_count"]])

METASPACE discovery:   0%|          | 0/3 [00:00<?, ?stage/s]

Current operation:   0%|          | 0/1 [00:00<?, ?operation/s]

Found 60 datasets


,dataset_id,name,analyzer_type,ionisation_source,mz_min,mz_max,pixel_count
0,2026-04-22_21h03m00s,kidney_test_metabolites_null_mz_shift_10_til_550,Orbitrap,MALDI,90.002395,999.994556,8575
1,2025-04-14_15h53m53s,00070_lgruber_qcl-msi_data11_glomeruli_wt_rms,timsTOF fleX,MALDI,300.015000,2500.000000,11057
2,2025-04-14_09h06m15s,00070_lgruber_qcl-msi_het-0717_data3_slide1-rms,timsTOF fleX,MALDI,300.006000,1999.970000,39149
3,2025-04-14_09h00m43s,00070_lgruber_qcl-msi_het-0707_data5_slide1-rms,timsTOF fleX,MALDI,300.006000,1999.980000,39021
4,2025-04-14_15h55m50s,00070_lgruber_qcl-msi_timson_glomeruli_data2_w...,timsTOF fleX,MALDI,300.010500,2500.000000,10163
5,2025-04-14_09h56m19s,00070_lgruber_qcl-msi_wt-4625_data6_slide1-rms,timsTOF fleX,MALDI,300.007500,1999.980000,47682
6,2025-04-14_10h18m43s,00070_lgruber_qcl-msi_wt-4628_data1_slide1-rms,timsTOF fleX,MALDI,300.007500,1999.990000,44832
7,2025-07-07_13h56m06s,mo6_metabolites,timsTOF fleX,MALDI,50.001500,649.990250,184011
8,2025-07-07_13h03m27s,msi2024013_20240909_multiomicsiii_nor_300-1350...,timsTOF fleX,MALDI,300.004500,1349.979750,83047
9,2025-07-07_10h55m54s,msi2024013_20240909_multiomicsii_nedc 50-650_f...,timsTOF fleX,MALDI,50.001500,649.990250,87617


## 2. Obiektywna detekcja duplikatów technicznych (metoda z `brain_dataset.ipynb`, rozszerzona)

Ta sama zasada: identyczny `pixel_count` **oraz** identyczne `mz_min`/`mz_max` (zaokrąglone do 3 miejsc) → prawie na pewno ten sam surowy skan zgłoszony ponownie.

**Rozszerzenie wynikające z audytu liver (patrz `liver_dataset_repaired.ipynb`, sekcja 2):** przy szerszym zastosowaniu tej reguły natrafiłem tam na fałszywie dodatni klaster — kilka datasetów o identycznym `pixel_count` i zakresie m/z, ale z nazwami różniącymi się identyfikatorem studzienki/płytki (`S3_W8` vs `S3_W4` itd.), czyli **różne fizyczne próbki zmierzone wg tego samego, ustalonego protokołu** (stały rozmiar siatki, stałe okno masy), a nie ten sam skan. Dlatego każdy znaleziony klaster dzielę teraz dodatkowo po tym, czy nazwy w klastrze **stają się identyczne** po usunięciu rozpoznanych tokenów technicznych (ppm, `_ML`/`_AQ`, `tic`, daty, "-total ion count", "- root mean square"):

- `high_confidence_duplicate` — nazwy zbiegają się do tego samego rdzenia → bezpiecznie wykluczam wszystkie poza jedną.
- `ambiguous_shared_template` — nazwy różnią się czymś, co nie jest rozpoznanym tokenem technicznym (np. inny identyfikator próbki/studzienki) → **nie wykluczam automatycznie**, zostawiam do przeglądu.

In [4]:
results["mz_min_r"] = results["mz_min"].round(3)
results["mz_max_r"] = results["mz_max"].round(3)
results["cluster_id"] = results.groupby(["pixel_count", "mz_min_r", "mz_max_r"]).ngroup()
cluster_size = results.groupby("cluster_id")["dataset_id"].transform("count")
results["is_duplicate_cluster"] = cluster_size > 1


def strip_technical_tokens(name: str) -> str:
    s = str(name).lower()
    s = re.sub(r"^\d{4}-\d{2}-\d{2}[_ ]", "", s)
    s = re.sub(r"^\d{8}_+", "", s)
    s = re.sub(r"[-_]?\d+ ?ppm\b", "", s)
    s = re.sub(r"\btic\b", "", s)
    s = re.sub(r"_(?:aq_ml|aq|ml)$", "", s)
    s = re.sub(r"-total ion count$", "", s)
    s = re.sub(r" - root mean square$", "", s)
    s = re.sub(r"[^a-z0-9]+", " ", s).strip()
    return s


results["name_residual"] = results["name"].apply(strip_technical_tokens)

cluster_confidence = {}
for cluster_id, group in results[results["is_duplicate_cluster"]].groupby("cluster_id"):
    residuals = set(group["name_residual"])
    cluster_confidence[cluster_id] = (
        "high_confidence_duplicate" if len(residuals) == 1 else "ambiguous_shared_template"
    )
results["cluster_confidence"] = results["cluster_id"].map(cluster_confidence)

TECH_SUFFIX_PENALTY = re.compile(r"(?:_ml$|_v$|ppm$|-total ion count$)", re.IGNORECASE)


def pick_keeper(group: "pd.DataFrame") -> str:
    scored = group.assign(
        penalty=group["name"].str.lower().str.contains(TECH_SUFFIX_PENALTY, regex=True).astype(int)
    )
    return scored.sort_values(["penalty", "dataset_id"]).iloc[0]["dataset_id"]


keepers = {
    cluster_id: pick_keeper(group)
    for cluster_id, group in results[results["cluster_confidence"] == "high_confidence_duplicate"].groupby("cluster_id")
}
results["duplicate_excluded"] = results.apply(
    lambda row: row.get("cluster_confidence") == "high_confidence_duplicate"
    and row["dataset_id"] != keepers[row["cluster_id"]],
    axis=1,
)

print("duplicate clusters found:", results["is_duplicate_cluster"].sum() and results.loc[results['is_duplicate_cluster'], 'cluster_id'].nunique())
print("confirmed (high-confidence) duplicate exclusions:", int(results["duplicate_excluded"].sum()))
display(
    results.loc[
        results["is_duplicate_cluster"],
        ["cluster_id", "dataset_id", "name", "pixel_count", "mz_min_r", "mz_max_r", "cluster_confidence", "duplicate_excluded"],
    ].sort_values(["cluster_confidence", "cluster_id"])
)

duplicate clusters found: 0
confirmed (high-confidence) duplicate exclusions: 0


,cluster_id,dataset_id,name,pixel_count,mz_min_r,mz_max_r,cluster_confidence,duplicate_excluded


### Wariant kalibracyjny `null_mz_shift` (ten sam wzorzec co w brain)

Dokładnie jak `granular_layer_mouse_brain_null_mz_shift_10_from_2575` w brain: `kidney_test_metabolites_null_mz_shift_10_til_550` ma identyczny `pixel_count` (8575) i identyczny `mz_max` co `kidney_test_metabolites`, a `mz_min` różni się o dokładnie 10 — zgodnie z nazwą. Różnica w `mz_min` (nie `mz_max`, jak w brain) nie zostałaby złapana zaokrągleniem powyżej, więc sprawdzam to osobno po nazwie, tak jak w `brain_dataset.ipynb`.

**To jest jedyny obiektywnie potwierdzony duplikat w całej puli kidney (60 datasetów) — i jest on obecnie częścią pobranego korpusu 30 datasetów (zobacz sekcję 4).**

In [5]:
results["mz_shift_qc_variant"] = results["name"].str.contains("null_mz_shift", case=False, na=False)
print("mz-shift QC variants:", int(results["mz_shift_qc_variant"].sum()))
display(
    results.loc[
        results["mz_shift_qc_variant"] | results["name"].eq("kidney_test_metabolites"),
        ["dataset_id", "name", "pixel_count", "mz_min", "mz_max"],
    ]
)

mz-shift QC variants: 1


,dataset_id,name,pixel_count,mz_min,mz_max
0,2026-04-22_21h03m00s,kidney_test_metabolites_null_mz_shift_10_til_550,8575,90.002395,999.994556
58,2019-03-19_17h21m15s,kidney_test_metabolites,8575,100.002395,999.994556


## 3. Morfologia i jakość

**Morfologia (Poziom 4):** dla kidney istotne mikrostruktury to kora (`cortex`), rdzeń (`medulla`), brodawka (`papilla`), miedniczka/kielich (`pelvis`/`calyx`), kłębuszek (`glomerul*`). W tej puli tylko dwa datasety jawnie wskazują fragment mikroanatomiczny: seria `glomeruli` (`00070_lgruber_qcl-msi_..._glomeruli_...`). Podobnie jak w brain, to kolumna doradcza — nie wykluczam automatycznie, bo w przeciwieństwie do brain (gdzie 4 fragmenty nazwałeś Ty sam), tutaj to wyłącznie moja heurystyka.

**Jakość / liczba pikseli (Poziom 2):** w brain próg 500 pikseli miał sens, bo tam pula zawierała jawne skany testowe (do 25 pikseli). W kidney **minimalna** liczba pikseli w całej puli 60 datasetów to 5025 — nie ma żadnych ekstremalnie małych akwizycji. Sztywny próg 500 przeniesiony z brain byłby więc pusty i nic by nie wnosił — nie stosuję go tutaj, bo żaden dataset kidney go nie łamie (sprawdzone poniżej, nie założone).

In [6]:
REGIONAL_TOKENS = re.compile(r"(?:cortex|medulla|papilla|pelvis|calyx|glomerul)", re.IGNORECASE)
results["morphology_hint"] = results["name"].apply(
    lambda n: "regional_or_microregion" if REGIONAL_TOKENS.search(str(n)) else "whole_section_likely"
)
print("regional/microregion hint count (heuristic, advisory):",
      int((results["morphology_hint"] == "regional_or_microregion").sum()))
display(results.loc[results["morphology_hint"] == "regional_or_microregion", ["dataset_id", "name", "pixel_count"]])

print("\npixel_count distribution (min shown to justify skipping a low-pixel flag):")
print(results["pixel_count"].describe())

regional/microregion hint count (heuristic, advisory): 2


,dataset_id,name,pixel_count
1,2025-04-14_15h53m53s,00070_lgruber_qcl-msi_data11_glomeruli_wt_rms,11057
4,2025-04-14_15h55m50s,00070_lgruber_qcl-msi_timson_glomeruli_data2_w...,10163



pixel_count distribution (min shown to justify skipping a low-pixel flag):
count        60.000000
mean      34795.583333
std       34910.788666
min        5025.000000
25%        8572.750000
50%       11258.000000
75%       61976.500000
max      184011.000000
Name: pixel_count, dtype: float64


## 4. Zestawienie z istniejącą, ręczną selekcją

`data/kidney_workspace/configs/datasets/kidney/filter.json` zawiera już 30 ręcznie wykluczonych ID z poprzedniej rundy przeglądu. Sprawdzam, czy obiektywny sygnał duplikatu (sekcja 2–2b) pokrywa się z tym, co człowiek już wykluczył, czy wskazuje coś **nowego**.

In [7]:
existing_filter = json.load(open("data/kidney_workspace/configs/datasets/kidney/filter.json"))
existing_selection = json.load(open("data/kidney_workspace/configs/datasets/kidney/selection.json"))
existing_excluded_ids = set(existing_filter.get("exclude_dataset_ids", []))
existing_selected_ids = set(existing_selection["dataset_ids"])
print(f"existing selection: {len(existing_selected_ids)} selected, {len(existing_excluded_ids)} manually excluded")

new_findings = set(
    results.loc[results["duplicate_excluded"] | results["mz_shift_qc_variant"], "dataset_id"]
)
missed_by_manual_review = new_findings & existing_selected_ids
already_caught = new_findings & existing_excluded_ids
print("objective findings already in the currently DOWNLOADED 30 (missed by manual review):", sorted(missed_by_manual_review))
print("objective findings already manually excluded before:", sorted(already_caught))

existing selection: 30 selected, 30 manually excluded
objective findings already in the currently DOWNLOADED 30 (missed by manual review): ['2026-04-22_21h03m00s']
objective findings already manually excluded before: []


## 5. Poprawiona ("repaired") konfiguracja

Unikam nadpisania istniejącego, używanego przez pipeline `kidney/filter.json` — piszę do równoległego katalogu `kidney_repaired/`, żebyś mógł porównać oba, zanim cokolwiek zamienisz. Zakres m/z zostaje **niezmieniony** (`200–900`, dokładnie jak dotychczas) — to, co się zmienia, to lista wykluczeń: unia dotychczasowych 30 ręcznych wykluczeń i nowo znalezionego obiektywnego duplikatu.

In [8]:
repaired_exclude_ids = sorted(existing_excluded_ids | new_findings)
print(f"exclude_dataset_ids: {len(existing_excluded_ids)} (existing) -> {len(repaired_exclude_ids)} (repaired)")

final_filters = {
    "organism": "Mouse",
    "organism_part": "Kidney",
    "polarity": "Negative",
    "condition": "Wildtype",
    "mz_min": 200,
    "mz_max": 900,
    "annotation_fdr": 0.1,
    "min_annotation_count": 1,
    "include_molecule_stats": True,
    "include_spatial_annotation_stats": False,  # see brain_dataset.ipynb section 6 for the cost rationale
    "exclude_dataset_ids": repaired_exclude_ids,
}
results_kidney_repaired = explorer.filter(final_filters)
print(f"repaired kidney shortlist: {len(results_kidney_repaired)} datasets (previously {len(existing_selected_ids)})")
display(results_kidney_repaired[["dataset_id", "name", "analyzer_type", "pixel_count", "molecule_count", "unique_molecule_count"]])

exclude_dataset_ids: 30 (existing) -> 31 (repaired)


METASPACE discovery:   0%|          | 0/3 [00:00<?, ?stage/s]

Current operation:   0%|          | 0/1 [00:00<?, ?operation/s]

repaired kidney shortlist: 29 datasets (previously 30)


,dataset_id,name,analyzer_type,pixel_count,molecule_count,unique_molecule_count
0,2025-07-06_22h07m04s,msi2024013_20240909_multiomicsii_nor 50-1350,timsTOF fleX,83704,103,44
1,2025-07-06_16h31m32s,msi2024013_20240909_multiomicsii_nedc 50-1350,timsTOF fleX,70784,23,13
2,2024-05-23_14h25m01s,K3 vs K6 neg -Jano,Orbitrap,20096,311,221
3,2024-04-10_17h06m19s,8223301,Orbitrap,20992,124,69
4,2024-02-20_01h55m56s,d28-2014-2,FTICR,7198,7,0
5,2024-02-20_01h57m32s,d28-2017-2,FTICR,7664,14,6
6,2024-02-20_01h54m01s,d14-2295,FTICR,5025,10,7
7,2024-02-20_01h54m41s,d28-2006-2,FTICR,8566,14,1
8,2024-02-20_01h53m31s,d14-2015,FTICR,6697,14,6
9,2024-02-20_01h49m35s,d14-2001-2,FTICR,8278,11,1


In [9]:
output_path = Path("data/kidney_workspace/configs/datasets/kidney_repaired")
exported = explorer.export_selection(output_path, sort_by="download_size_bytes", ascending=False)
exported

{'filters': PosixPath('data/kidney_workspace/configs/datasets/kidney_repaired/filter.json'),
 'selection': PosixPath('data/kidney_workspace/configs/datasets/kidney_repaired/selection.json')}

## Podsumowanie

- Szeroka pula (Wildtype, bez filtra m/z): 60 datasetów.
- Obiektywne duplikaty techniczne (identyczny `pixel_count`+`mz_min`+`mz_max`): **0** klastrów w tej puli (dane kidney są w większości pojedynczymi, unikalnymi akwizycjami zwierząt `d1`–`d28`/`con`/`c2` — inaczej niż brain, gdzie reprocessing pipeline'y generowały wiele wariantów tego samego skanu).
- Wariant kalibracyjny `null_mz_shift`: **1**, i jest on obecnie w pobranym korpusie 30 — nowe, nieznalezione wcześniej ustalenie.
- Fragmenty mikroanatomiczne (heurystyka, `glomeruli`): 2, tylko doradczo.
- Niska liczba pikseli: brak przypadków — próg z brain nie ma tu zastosowania.
- Eksport do `data/kidney_workspace/configs/datasets/kidney_repaired/` — istniejący `kidney/` **nie został nadpisany**.

Analiza wspólnego zakresu m/z między narządami — świadomie pominięta, do rozwiązania osobno.